# Extract & Transform - Différentes parties :
1. Extract : 
    - extraction des données Existants de l'ADEME
    - extraction des données Neufs de l'ADEME
    - extraction de données supplémentaires (températures)

2. Transform :
    - Transformation des données (coordonnéees) pour joindre les températures
    - Transformation de l'année de construction en classe
    - Transformation des variables "type_generateur_chauffage_principal" et "type_generateur_n1_ecs_n1" en classes plus larges


## 1. Extract

### Extraction des existants de l'ADEME

On sélectionne les variables pour existant

In [1]:
SELECT = [
    "etiquette_dpe", "conso_5_usages_ef",
    "type_batiment", "surface_habitable_logement", "nombre_niveau_logement", "hauteur_sous_plafond", "annee_construction",
    "adresse_ban", "zone_climatique", "classe_altitude", "type_installation_chauffage", "type_generateur_chauffage_principal",
    "type_energie_principale_chauffage", "type_installation_ecs", "type_generateur_n1_ecs_n1",
    "isolation_toiture", "qualite_isolation_plancher_bas", "qualite_isolation_murs",
    "type_generateur_froid",
    "modele_dpe", "version_dpe", "_geopoint"
]

On requête l'API à l'aide de la classe API_ADEME

In [ ]:
# get all existant
import pandas as pd

from Classes_API.API_ADEME import API_ADEME

test = API_ADEME()

params = {"select": SELECT,
          "size": 10000
          }

data = test.get_data(print_progress=True, **params)

data_df = pd.DataFrame(data)

# on écrit le dataframe obtenu :
data_df.to_csv("data/existants_ALL.csv", sep=",", index=False)

# différents temps pour le bloc de code:
# 44min 28.6 sec
# 43min 31.9 sec
# 42min 22.0 sec
# 42min 37.0 sec

total à atteindre : 1544750
10000 / 1544750
20000 / 1544750
30000 / 1544750
40000 / 1544750
50000 / 1544750
60000 / 1544750
70000 / 1544750
80000 / 1544750
90000 / 1544750
100000 / 1544750
110000 / 1544750
120000 / 1544750
130000 / 1544750
140000 / 1544750
150000 / 1544750
160000 / 1544750
170000 / 1544750
180000 / 1544750
190000 / 1544750
200000 / 1544750
210000 / 1544750
220000 / 1544750
230000 / 1544750
240000 / 1544750
250000 / 1544750
260000 / 1544750
270000 / 1544750
280000 / 1544750
290000 / 1544750
300000 / 1544750
310000 / 1544750
320000 / 1544750
330000 / 1544750
340000 / 1544750
350000 / 1544750
360000 / 1544750
370000 / 1544750
380000 / 1544750
390000 / 1544750
400000 / 1544750
410000 / 1544750
420000 / 1544750
430000 / 1544750
440000 / 1544750
450000 / 1544750
460000 / 1544750
470000 / 1544750
480000 / 1544750
490000 / 1544750
500000 / 1544750
510000 / 1544750
520000 / 1544750
530000 / 1544750
540000 / 1544750
550000 / 1544750
560000 / 1544750
570000 / 1544750
580000 / 154

On sélectionne les variables pour neuf :

on enlève la variable annee_construction car elle n'existe pas pour les logements neufs

In [3]:
SELECT.remove("annee_construction")

In [ ]:
# get all neuf
import pandas as pd

from Classes_API.API_ADEME import API_ADEME

test = API_ADEME()

params = {"select": SELECT,
          "size": 10000
          }

data = test.get_data(neuf = True, print_progress=True, **params)

data_df = pd.DataFrame(data)
data_df.to_csv("data/neufs_ALL.csv", sep=",", index=False)

# différents temps pour le bloc de code:
# 3min 47.9 sec
# 3min 42.4 sec
# 3 min 0.0 sec
# 3 min 2.3 sec

total à atteindre : 159629
10000 / 159629
20000 / 159629
30000 / 159629
40000 / 159629
50000 / 159629
60000 / 159629
70000 / 159629
80000 / 159629
90000 / 159629
100000 / 159629
110000 / 159629
120000 / 159629
130000 / 159629
140000 / 159629
149629 / 159629


#### Fusion des jeux de données

In [ ]:
import pandas as pd

data_existant = pd.read_csv("data/existants_ALL.csv", sep = ",")
data_neuf = pd.read_csv("data/neufs_ALL.csv", sep = ",")


On rajoute la colonne manquante dans le jeu de données des logements neufs : l'année de construction

In [ ]:
data_neuf["annee_construction"] = 2025

Merge des jeux de données :

In [ ]:
data_total = pd.concat([data_neuf, data_existant], ignore_index=True)
# on ignore les index car on ne les utilise jamais, et nous n'allons pas les sauvegarder ensuite.

On sauvegarde le jeu de données pré-transformations

In [ ]:
data_total.to_csv("data/ADEME_total.csv", sep = ",", index=False)

### Extractions des températures

On requête les API de températures avec la classe API_TEMPERATURE

In [ ]:
import pandas as pd
from Classes_API.API_TEMPERATURE import API_TEMPERATURE

test = API_TEMPERATURE()
SELECT = ["NUM_POSTE", "NOM_USUEL", "LAT", "LON", "ALTI", "AAAAMM", "TX", "TN"]
params = {
    "columns": SELECT,
    "page_size":200
}

data_temperature = test.get_data(print_progress=True, **params)

# on sauvegarde les données dans le fichier température.csv
data_temperature.to_csv("data/temperature.csv", sep=",", index=False)

[DEBUG] API_TEMPERATURE._request() → params={'columns': 'NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMM,TX,TN', 'page_size': 200} | status=200
[DEBUG] API_TEMPERATURE._get_content() → total=N/A | dims=(200, 8)
[DEBUG] API_TEMPERATURE._get_next() → next_link=https://tabular-api.data.gouv.fr/api/resources/f292971a-dd3c-4d76-9e52-c265e2f909a5/data/?columns=NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMM,TX,TN&page=2&page_size=200
[DEBUG] API_TEMPERATURE._get_total() → total=351
Pour le département :01, total à atteindre : 351
[DEBUG] API_TEMPERATURE._request() → params=None | status=200
[DEBUG] API_TEMPERATURE._get_length() → length=151
[DEBUG] API_TEMPERATURE._get_content() → total=N/A | dims=(151, 8)
[DEBUG] API_TEMPERATURE._get_length() → length=151
dep:01 : 151 / 351
[DEBUG] API_TEMPERATURE._get_next() → next_link=None
[DEBUG] API_TEMPERATURE._request() → params={'columns': 'NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMM,TX,TN', 'page_size': 200} | status=200
[DEBUG] API_TEMPERATURE._get_content() → total=

## Transform

Transformation des coordonnées des jeux de données ADEME et des températures pour pouvoir réaliser une jointure spatiale.

In [ ]:
data_total["_geopoint"].isnull().sum()
# on peut voir qu'il n'existe pas de valeur nulles dans les coordonnées de l'ADEME

np.int64(0)

Pour les valeurs aberrantes dans les coordonnées :

Si elles ont une adresse, alors on récupère les coordonnées via un API de géocodage

Autrement, on les enlève du jeu de données

In [ ]:
print(len(data_total))
data_total.dropna(subset=["adresse_ban"], inplace=True)
print(len(data_total))
# il y a environ 2000 données qui n'ont pas d'adresses.

1704379
1702447


On crée les colonnes _lat et _lon, en coupant la colonne _geopoint

In [ ]:
data_total[["_lat", "_lon"]] = data_total["_geopoint"].str.split(",", expand=True).astype(float)

On décide de vérifier les valeurs aberrantes sur les latitude et longitude via une bounding box autour de la région Auvergne Rhône Alpes

In [ ]:
bornes_AURA = [
    [44.005, 46.925],  # latitude minimale et maximale
    [1.908, 7.32]      # longitude minimale et maximale
]

In [ ]:
from Transformation_donnees.detect_outliers_via_bornes import detect_outliers_via_bornes

data_total = detect_outliers_via_bornes(data_total, cols=["_lat", "_lon"], bornes=bornes_AURA, multiple_cols=False)
# on précise multiple_cols = False car on veut rajouter une seule colonne
# qui précise si la ligne possède des coordonnées aberrantes

On récupère les lignes correspondantes

In [ ]:
from Transformation_donnees.get_line_index_of_outliers import get_line_index_of_outliers

index_des_adresses = get_line_index_of_outliers(data_total, cols=["_outlier"])


Il suffit maintenant de récupérer les adresses de ces lignes

In [ ]:
liste_d_adresses = data_total.loc[index_des_adresses, "adresse_ban"]

On requête maintenant un API de géocodage (NOMINATIM) via la classe API_NOMINATIM

In [ ]:
import pandas as pd
from Classes_API.API_NOMINATIM import API_NOMINATIM


L = liste_d_adresses.astype(str).tolist()
test = API_NOMINATIM()
data = test.get_data(liste_adresses = L)


df_lat_lon_complementaire = pd.DataFrame(data)

test
[DEBUG] API_NOMINATIM.get_data() → Aucun résultat pour 30 Avenue du Commandant Madeline 63200 Riom
{'adresse': 'Route de Bourg-En-Bresse 01851 Marboz', 'lat': '46.3052829', 'lon': '5.2360028'}
{'adresse': '15 Place Jean Jaurès 42000 Saint-Étienne', 'lat': '45.4418644', 'lon': '4.3856606'}
{'adresse': "38 Rue d'Arcole 42000 Saint-Étienne", 'lat': '45.4400214', 'lon': '4.3813672'}
{'adresse': '34 Route de Bort 15190 Condat', 'lat': '45.3457054', 'lon': '2.7330471'}
{'adresse': '1bis Chemin de la Planta 42680 Saint-Marcellin-en-Forez', 'lat': '45.4716399', 'lon': '4.1825534'}
{'adresse': '18 Rue Gambetta 42230 Roche-la-Molière', 'lat': '45.4331159', 'lon': '4.3253256'}
{'adresse': '116 Rue Antoine Dupuy 42510 Bussières', 'lat': '45.8337476', 'lon': '4.2710350'}
{'adresse': '16 Rue de Boiron 42290 Sorbiers', 'lat': '45.4858615', 'lon': '4.4433354'}
{'adresse': '4 Chemin des Roches 63460 Beauregard-Vendon', 'lat': '45.9587642', 'lon': '3.1084014'}
{'adresse': '12 Rue Beaumarchais 42100

On rajoute les index qui n'ont pas pu suivre le rajout des données

In [ ]:
df_lat_lon_complementaire = (
    liste_d_adresses
    .rename("adresse")
    .reset_index()  # garde l'index d'origine dans une colonne
    .merge(df_lat_lon_complementaire, on="adresse", how="left")
)
df_lat_lon_complementaire.index = df_lat_lon_complementaire["index"]
df_lat_lon_complementaire.drop(columns="index", inplace = True)

On transforme le nom des colonnes et on enlève la colonne adresse pour réaliser une mise à jour du jeu de données principal

In [ ]:
df_replace_renamed = df_lat_lon_complementaire.rename(columns={'lat': '_lat', 'lon': '_lon'})
df_replace_renamed.drop(columns="adresse", inplace=True)

Mise à jour

In [ ]:
data_total.update(df_replace_renamed)

C:\Users\olivi\AppData\Local\Temp\ipykernel_12980\3580274263.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[45.566999975498284 46.08334802625607 44.94531599893682 ...
 45.79084900685032 45.82013299704334 46.3319799609564]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data_total.update(df_replace_renamed)
C:\Users\olivi\AppData\Local\Temp\ipykernel_12980\3580274263.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[5.912342029197113 6.411200032158504 4.893047967578588 ...
 3.2612540439818125 6.727940026320779 6.042545996473962]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data_total.update(df_replace_renamed)


### Rajout des températures dans le jeu de données de l'ADEME

L'idée de la jointure est celle-ci :

Une jointure via le calcul de la distance entre deux coordonnées géographiques est couteuse en calcul et donc en temps, nous l'avons experimenté et nous avons décidé de plutôt utiliser cKDTree. 

On réalise donc un arbre de recherche avec cKDTree sur les données des stations météos. Ensuite, on utilise cet arbre pour trouver le point le plus proche des coordonnées de la ligne du jeu de données de l'ADEME. Puis, on rajoute les valeurs qui nous intéresse avec l'index trouvé.

In [ ]:
from scipy.spatial import cKDTree

tree = cKDTree(data_temperature[['LAT', 'LON']].values)
_, indices = tree.query(data_total[['_lat', '_lon']].values, k=1)


data_total['TX'] = data_temperature.iloc[indices]['TX'].values
data_total['TN'] = data_temperature.iloc[indices]['TN'].values

### Rajout de la classe de la période de construction : periode_construction

In [ ]:
bins = [0, 1960, 1970, 1980, 1990, 2000, 2010, float('inf')]
labels = [
    'Avant 1960',
    '1961 - 1970',
    '1971 - 1980',
    '1981 - 1990',
    '1991 - 2000',
    '2001 - 2010',
    'Après 2010'
]

data_total['periode_construction'] = pd.cut(data_total['annee_construction'], bins=bins, labels=labels, right=True)


### Transformation des variables "type_generateur_chauffage_principal" et "type_generateur_n1_ecs_n1" en classes

Nous avons utilisé une approche similaire "type_generateur_chauffage_principal" et "type_generateur_n1_ecs_n1. Nous avons d'abord observé les classes déjà existantes, puis nous avons décidé de quoi regrouper en classes plus larges. Nous avons aussi créé un algorithme pour "proposer" différentes classes selon la ressemblance sémantique, ici non utilisé, mais disponible ici : fonctions_supplementaires/proposer_classes_generiques.py. Nous avons aussi exploré une approche NLP non supervisé, aussi non utilisé, et disponible ici : fonctions_supplementaires/regrouper_termes_via_semantique.

Enfin, nous avons décidé d'utiliser les distance TF-IDF pour calculer l'appartenance aux classes que nous apportons. En effet, ces colonnes sont déjà des classes, mais trop large (environ 150 classes dans notre jeu de données), surtout pour l'utilisateur de notre modèle de prédiction. Nous résumons donc les classes dans des plus larges.

#### Pour type_generateur_chauffage_principal

On choisit les différents groupes qu'on voudrait obtenir

In [ ]:
classes_type_generateur_chauffage_principal = ["Chaudière gaz", "Chaudière fioul", "Chaudière bois", "Chaudière charbon", "Chaudière électrique", "Chaudière gpl/propane/butane", "pompe à chaleur hybride", "Convecteur électrique", "Poêle / Cuisinière / Foyers / insert flamme verte", "autre système / émetteurs", "radiateur à gaz", "réseau de chaleur", "radiateur électrique", "PAC géothermique", "Plancher ou plafond rayonnant électrique", "convecteur bi-jonction", "Réseau de chaleur", "PAC air/eau - Pompe à chaleur", "non spécifié"]


On remplace les NA par "non spécifié" pour créer une classe de NA

In [ ]:
data_total["type_generateur_chauffage_principal"].fillna(value = "non spécifié", inplace=True)

C:\Users\olivi\AppData\Local\Temp\ipykernel_12980\830576453.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_total["type_generateur_chauffage_principal"].fillna(value = "non spécifié", inplace=True)


In [ ]:
from Transformation_donnees.regrouper_colonne_en_classes import regrouper_colonne_en_classes
from Transformation_donnees.nettoyage import nettoyer


colonne = pd.Series(data_total["type_generateur_chauffage_principal"])
df_chauffage_principal = regrouper_colonne_en_classes(colonne, classes_type_generateur_chauffage_principal, cleaner=nettoyer)


Nettoyage des textes uniques...
Encodage TF-IDF pour 150 valeurs uniques...
Calcul des similarités cosinus...
fonction terminée.


On remplace les valeurs dans la colonne par les classes auxquelles elles appartiennent

In [ ]:
mapping_classes = dict(zip(df_chauffage_principal["texte"], df_chauffage_principal["classe_lisible"]))

data_total["type_generateur_chauffage_principal"] = data_total["type_generateur_chauffage_principal"].map(
    lambda x: mapping_classes.get(x, x)
)

#### Pour type_generateur_n1_ecs_n1

On choisit les différents groupes qu'on voudrait obtenir

In [ ]:
classes_type_generateur_n1_ecs_n1 = ["Chaudière gaz", "Chaudière fioul", "Chaudière bois", "Chaudière charbon", "PAC / pompe à chaleur", "Chaudière gpl/propane/butane", "CET sur air", "Chauffe-eau gaz", "chauffe-eau électrique", "Réseau de chaleur", "chaudière condensation", "Accumulateur gaz", "autre système / émetteurs", "Poêle / Cuisinière / Foyers / insert flamme verte", "Ballon électrique", "non spécifié"]


On remplace les NA par "non spécifié" pour créer une classe de NA

In [ ]:
data_total["type_generateur_n1_ecs_n1"].fillna(value = "non spécifié", inplace=True)


C:\Users\olivi\AppData\Local\Temp\ipykernel_12980\2315908521.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_total["type_generateur_n1_ecs_n1"].fillna(value = "non spécifié", inplace=True)


In [ ]:
colonne = pd.Series(data_total["type_generateur_n1_ecs_n1"])
df_generateur_n1 = regrouper_colonne_en_classes(colonne, classes_type_generateur_n1_ecs_n1, cleaner=nettoyer)


Nettoyage des textes uniques...
Encodage TF-IDF pour 127 valeurs uniques...
Calcul des similarités cosinus...
fonction terminée.


On remplace les valeurs dans la colonne par les classes auxquelles elles appartiennent

In [ ]:
mapping_classes = dict(zip(df_generateur_n1["texte"], df_generateur_n1["classe_lisible"]))

data_total["type_generateur_n1_ecs_n1"] = data_total["type_generateur_n1_ecs_n1"].map(
    lambda x: mapping_classes.get(x, x)
)

Enfin, on enlève les colonnes inutiles et on exporte les données

In [ ]:
cols_to_drop = ["_geopoint", "annee_construction", "_lat", "_lon"]
data_total.drop(columns=cols_to_drop, inplace=True)

In [ ]:
data_total.to_csv("data/donnees_finales.csv", sep=",", encoding="UTF-8", index=False)